In [ ]:
import pandas as pd
from etl.enrichment import *
from etl.embeddings import *
from etl.duckdb_handler import *
from etl.utils import *
from etl.cleaningdata import *
import warnings
warnings.filterwarnings("ignore")
import polars as pl
import json
from etl.data_validation import *




In [3]:
# Load Data

df = pd.read_csv("data/input_data/tech_news.csv")
metadata = pd.read_json("data/input_data/company_metadata.json",  orient='index').reset_index()
metadata.rename(columns={'index': 'company_name'}, inplace=True)



In [ ]:
# Clean  and normalize data
df["revenue_usd"] = df["revenue"].apply(parse_revenue)
df["published_date"] = df["published_date"].apply(normalize_date)
df = extract_date_parts(df)
df["category"] = df["category"].apply(normalize_category)

In [5]:
# Company Name Validation
df = validate_company_names(df, metadata)

In [6]:
 # Data enrichment: join two dataframes and adding new calculated columns
df = df.merge(metadata, on="company_name", how="left")

df["company_age"] = df.apply(
    lambda x: compute_company_age(x["founded_year"], x["published_date"]), axis=1
)
df["company_size_category"] = df["employee_count"].apply(company_size_category)

In [ ]:
# Embeddings 
df = add_embeddings(df)

In [ ]:
# Similarity 
df = add_top_similar(df)

In [ ]:
# Add to DuckDB
con=load_to_duckdb(df)

query_result = con.sql("""
    SELECT *
    FROM articles
    WHERE revenue_usd >= 50000000
    AND year BETWEEN 2022 AND 2024
    AND (category = 'AI_ML' OR industry = 'AI/ML')
""")

# Fetch results as a Pandas DataFrame
print(query_result.df())
con.close()

In [11]:
#Filter
df = filter_ai_articles(df)


In [12]:
# Export clean data
export_csv(df)

In [13]:
# similar article search:

find_similar_articles("AI startup funding", df, 5)

,article_id,similarity
50,ART0051,0.599743
369,ART0370,0.599743
218,ART0219,0.580880
358,ART0359,0.580777
321,ART0322,0.572589


In [43]:

print(df["published_date"].dtype)


datetime64[us, UTC]


In [ ]:
con = duckdb.connect("articles.db")
query = """
    SELECT *
    FROM articles
    WHERE revenue_usd >= 50000000
    AND year BETWEEN 2022 AND 2024
    AND (category = 'AI_ML' OR industry = 'AI/ML')
"""
result = query_duckdb(con,query)
print(result)
con.close()

In [ ]:

con = duckdb.connect("articles.db")
query_result = con.sql("""
    SELECT *
    FROM articles
    WHERE revenue_usd >= 50000000
    AND year BETWEEN 2022 AND 2024
    AND (category = 'AI_ML' OR industry = 'AI/ML')
""")

# Fetch results as a Pandas DataFrame
print(query_result.df())
con.close()


In [ ]:
df.dtypes


article_id                               str
title                                    str
company_name                             str
published_date           datetime64[us, UTC]
category                                 str
revenue                                  str
summary                                  str
url                                      str
author                                   str
word_count                           float64
revenue_usd                            int64
year                                   int32
month                                  int32
quarter                                  str
matched_company                          str
company_valid                           bool
founded_year                           int64
headquarters                             str
employee_count                         int64
industry                                 str
is_public                               bool
stock_ticker                             str
company_ag